In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def probablity(x, data):
    data=np.array(data)
    if len(data) == 0:
        return 0
    return np.sum(data==x)/len(data)

In [ ]:
def entropy(y):
    classes = np.unique(y)
    ent = 0
    
    for c in classes:
        p = probablity(c, y)
        if p > 0:
            ent += p * np.log(p)
        
    return -ent



def information_gain(x, y, feature, impurity='gini'):
    if impurity=='gini':
        parent_entropy = gini_impurity(y)
    
        values = np.unique(x[:, feature])
        weighted_entropy = 0
        
        for v in values:
            subset_y = y[x[:, feature] == v]
            weight = len(subset_y) / len(y)
            weighted_entropy += weight * gini_impurity(subset_y)
        
        return parent_entropy - weighted_entropy
    
    elif impurity=='entropy':
        parent_entropy = entropy(y)
        
        values = np.unique(x[:, feature])
        weighted_entropy = 0
        
        for v in values:
            subset_y = y[x[:, feature] == v]
            weight = len(subset_y) / len(y)
            weighted_entropy += weight * entropy(subset_y)
        
        return parent_entropy - weighted_entropy

    else:
        raise ValueError('Invalid impurity!')


def gini_impurity(y):
    if len(y)==0:
        return 0

    classes=np.unique(y)
    gi=0

    for c in classes:
        gi+=probablity(c,y)**2

    return 1-gi

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_samples = 20000

# Features
age = np.random.randint(21, 65, n_samples)
income = np.random.normal(50000, 15000, n_samples).astype(int)
credit_score = np.random.randint(300, 850, n_samples)
loan_amount = np.random.normal(20000, 8000, n_samples).astype(int)
years_employed = np.random.randint(0, 40, n_samples)

# Target logic (realistic-ish rules)
loan_approved = (
    (credit_score > 650) &
    (income > 40000) &
    (loan_amount < 0.4 * income) &
    (years_employed > 2)
).astype(int)

# Add some noise (real-world imperfection)
noise = np.random.rand(n_samples) < 0.1
loan_approved = np.where(noise, 1 - loan_approved, loan_approved)

# Create DataFrame
df = pd.DataFrame({
    'age': age,
    'annual_income': income,
    'credit_score': credit_score,
    'loan_amount': loan_amount,
    'years_employed': years_employed,
    'loan_approved': loan_approved
})

print(df.head())
print(df['loan_approved'].value_counts())

In [ ]:
df

In [ ]:
def build_tree(x, y, features, depth, impurity_function='gini'):
    if len(set(y)) == 1:
        return y[0]
    
    if depth==0 or len(features)==0:
        return max(set(y), key=list(y).count)
    
    igs= [information_gain(x, y, feature, impurity_function) for feature in features]
    feature_ig= dict(zip(features, igs))
    best_feature= max(feature_ig, key=feature_ig.get)

    tree= {best_feature: {}}

    values=x[best_feature].unique()

    for v in values:
        subset=x[x[best_feature]==v]
        sub_y=y[subset.index]

        if len(subset)==0:
            tree[best_feature][v]=max(set(y), key=list(y).count)
        else:
            subtree=build_tree(
                subset.drop(best_feature),
                sub_y,
                features.drop(best_feature),
                depth-1,
                impurity_function
            )

            tree[best_feature][v]=subtree
    return tree

In [ ]:
def decision_tree_classifier(dataset, unseen_data, builder='gini', height=5):
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    features=dataset.columns[:-1]

    split=int(0.7*len(dataset))

    train_data=dataset.iloc[:split]
    test_data=dataset.iloc[split:]
    
    x_train=train_data.iloc[:,:-1]
    y_train=train_data.iloc[:,-1].values

    x_test=test_data.iloc[:,:-1]
    y_test=test_data.iloc[:,-1].values


    if builder=='gini':
        impurity_function=gini_impurity
    elif builder=='entropy':
        impurity_function=entropy
    else:
        raise ValueError('Invalid builder function chosen!')
    
    decision_tree=build_tree(x_train,
                             y_train,
                             features,
                             height,
                             impurity_function
                            )
